# Air Quality Across India: Big Data Analysis of CPCB Data (2010-2023)

**Big Data Analytics, MSc Data Analytics**
Abhroneel Mukherjee (PRN 25060242005) and Meetanshi Gaba (PRN 25060242083)

This notebook is the whole project in one place, so we can walk through the code during the presentation. It follows the same order as the slides:

1. Setting up Spark in Colab
2. Getting the CPCB data
3. A first look at the raw files (and why they are messy)
4. **Ingest**: 453 files with 198 different layouts become one table (Bronze)
5. **Clean**: seven cleaning rules, each one logged (Silver)
6. **AQI**: the official CPCB daily Air Quality Index (Gold)
7. **Spark SQL**: answering our five research questions, with charts
8. **Spark RDDs**: the same counts done the low-level way
9. **MapReduce**: a Python mapper and reducer in the Hadoop Streaming style
10. **Hive**: external and managed tables, Parquet vs ORC, a Python UDF
11. Why Parquet: CSV vs Parquet speed, and reading Spark's query plan

On our laptop the raw files sat in HDFS on a single-node Hadoop cluster. Colab has no Hadoop cluster, so here the same Spark code reads from Colab's local disk instead. The only thing that changes is the path (`hdfs://localhost:9000/cpcb/raw` on the laptop, `/content/cpcb_raw` here). Section 9 and 10 show the Hadoop commands we ran on the cluster.

**How to run:** `Runtime > Run all`. The full dataset took about 7 minutes in a test on 2 CPU cores (the size of a free Colab machine), download included; give it 10-15 minutes on Colab to be safe. If you only want a quick demo, set `QUICK_RUN = True` in the settings cell below and it uses three states only (a few minutes). The numbers in the text are from the full run.

## 1. Setup

### 1.1 Settings

Everything you might want to change is in this one cell. `QUICK_RUN` picks a small subset of stations by their state code (every station file is named like `DL001.csv`, where `DL` is Delhi).

In [ ]:
import os, sys, glob, time

QUICK_RUN = False                       # True = only the states below, for a fast demo
QUICK_STATES = ["DL", "KA", "WB"]       # Delhi, Karnataka, West Bengal

IN_COLAB = "google.colab" in sys.modules
BASE = "/content" if IN_COLAB else os.path.abspath("colab_work")
DATA_DIR = f"{BASE}/cpcb_raw"           # raw CSVs go here (on the laptop this was HDFS /cpcb/raw)
LAKE_DIR = f"{BASE}/cpcb_lake"          # our Bronze / Silver / Gold Parquet layers
os.makedirs(BASE, exist_ok=True)
print("Running in Colab:", IN_COLAB, "| working folder:", BASE)

### 1.2 Install PySpark and Java

Spark is written in Scala and runs on the Java Virtual Machine, so it needs Java. Spark 4 needs Java 17 or newer, and Colab does not always ship with that, so we check first and install it only if it is missing. `pip install pyspark` gives us Spark itself, including the Hive libraries we use later.

Lines that start with `!` are shell commands, not Python. Colab runs them in a terminal for us.

In [ ]:
import subprocess

def java_major():
    # "java -version" prints to stderr, e.g.  openjdk version "17.0.12" ...
    out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    try:
        return int(out.split('"')[1].split(".")[0])
    except Exception:
        return 0

if IN_COLAB:
    !pip -q install pyspark==4.0.1
    if java_major() < 17:
        !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
        os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("Java version:", java_major())

### 1.3 Start a Spark session

The `SparkSession` is our entry point to Spark. Everything (DataFrames, SQL, RDDs, Hive) goes through this one object.

- `master("local[*]")` means run Spark on this machine and use every CPU core. On a real cluster this would point at YARN instead.
- `spark.driver.memory` is how much RAM Spark may use. Colab gives about 12 GB in total, so we take 6 GB.
- `shuffle.partitions` is how many pieces Spark splits data into when it has to regroup it (for a `GROUP BY` or a join). The default of 200 is too many for a small machine.
- `adaptive.enabled` lets Spark adjust its plan while the query runs, for example by merging tiny partitions.
- The time zone is fixed to UTC so Spark never shifts our timestamps. The readings are already in Indian time.
- `enableHiveSupport()` turns on the Hive catalog, which we use in Section 10.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F          # F.col, F.avg, F.when ... all the column functions
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType

spark = (SparkSession.builder
         .appName("cpcb-air-quality")
         .master("local[*]")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", "24")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.parquet.compression.codec", "snappy")
         .config("spark.ui.showConsoleProgress", "false")          # no progress bars in the output
         .config("spark.sql.warehouse.dir", f"{LAKE_DIR}/hive_warehouse")
         .config("spark.driver.extraJavaOptions", f"-Dderby.system.home={BASE}")
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")   # hide Spark's very chatty INFO messages
sc = spark.sparkContext                  # the older, lower-level entry point, used for RDDs later
print("Spark", spark.version, "| cores:", sc.defaultParallelism)

A small helper we use throughout: it times a block of code, so we can say how long each step took.

In [ ]:
from contextlib import contextmanager

@contextmanager
def timed(label):
    start = time.perf_counter()
    yield                                    # the code inside the "with" block runs here
    print(f"[time] {label}: {time.perf_counter() - start:.1f}s")

## 2. Get the data

The dataset is **Time Series Air Quality Data of India (2010-2023)** on Kaggle, compiled from the CPCB (Central Pollution Control Board) portal. It is a 465 MB zip that unpacks to 1.7 GB of CSVs: one file per monitoring station, plus `stations_info.csv` that says which state and city each station is in.

Kaggle lets anyone download public datasets through its API URL, so we do not need a Kaggle account here. If the files are already there, this cell skips the download.

In [ ]:
URL = "https://www.kaggle.com/api/v1/datasets/download/abhisheksjha/time-series-air-quality-data-of-india-2010-2023"

if not os.path.exists(f"{DATA_DIR}/stations_info.csv"):
    !curl -sL -o {BASE}/archive.zip "{URL}"
    !unzip -q -o {BASE}/archive.zip -d {DATA_DIR}
    !rm -f {BASE}/archive.zip

!ls {DATA_DIR} | head -5
!du -sh {DATA_DIR}

Now we build the list of station files we will work with. `glob` finds files by pattern: `[A-Z]*.csv` means "starts with a capital letter", which picks every station file (`AP001.csv`, `DL001.csv`, ...) and skips `stations_info.csv`.

In [ ]:
STATIONS_CSV = f"{DATA_DIR}/stations_info.csv"
station_files = sorted(glob.glob(f"{DATA_DIR}/[A-Z]*.csv"))
if QUICK_RUN:
    station_files = [f for f in station_files if os.path.basename(f)[:2] in QUICK_STATES]

raw_bytes = sum(os.path.getsize(f) for f in station_files)
print(f"{len(station_files)} station files, {raw_bytes / 1e9:.2f} GB")

## 3. First look at the raw files

### 3.1 One file on its own

Let's load a single station (DL001, in Delhi) the easy way: `header=True` uses the first line as column names, and `inferSchema=True` makes Spark guess the data types by scanning the file.

In [ ]:
one = spark.read.csv(f"{DATA_DIR}/DL001.csv", header=True, inferSchema=True)
one.printSchema()
print("rows:", one.count(), "| columns:", len(one.columns))
one.show(5)

For a single file that works fine. Each row is one hour at one station: a `From Date`, a `To Date`, and one column per pollutant or weather reading, with the unit in brackets.

### 3.2 The problem: the files do not agree with each other

We read just the first line (the header) of every file and count how many *different* headers there are. Python's `csv.reader` splits the line properly, even when a value has a comma inside quotes.

We group files that share an identical header into a dictionary: `{header: [list of files]}`. `defaultdict(list)` is a dictionary that starts every new key with an empty list, so we can `.append` straight away.

In [ ]:
import csv
from collections import defaultdict

def read_header(path):
    with open(path, encoding="utf-8") as fh:
        return tuple(next(csv.reader(fh)))     # tuple so it can be used as a dictionary key

groups = defaultdict(list)
for path in station_files:
    groups[read_header(path)].append(path)

print(len(groups), "different header layouts across", len(station_files), "files\n")
for header in list(groups)[:3]:
    print(len(header), "columns:", ", ".join(header[:8]), "...")

On the full data this gives **198 different layouts**. Columns appear in different orders, some stations have extra sensors, and some use different units (CO in mg/m3 at most stations but in ug/m3 or ng/m3 at a few).

If we gave Spark all 453 files with `header=True`, it would take the *first* file's header and apply it by position to every other file. A PM10 value could end up in the NO2 column. So we need our own ingestion step.

On the laptop we did this same header scan through the Hadoop FileSystem API, because the files lived in HDFS. In Colab they are ordinary files, so plain `open()` is enough.

## 4. Ingest: from 198 layouts to one table (Bronze)

The plan:
1. For every raw column name like `PM2.5 (ug/m3)`, work out a standard name (`pm25`) and a factor that converts its unit into our standard unit.
2. Read each layout group with its own schema, so columns line up correctly.
3. Stack all the groups into one table by column *name* and save it as Parquet. This first clean-ish copy is called **Bronze**.

### 4.1 The mapping rules

`PARAM_MAP` turns CPCB's labels into short names. `CANONICAL_UNIT` is the unit we store each pollutant in (following CPCB's own convention). `CONVERSIONS` holds the factor for every other unit we found. Gas values in ppb are converted to ug/m3 using molecular weight / 24.45, which is the standard conversion at 25 C and normal air pressure.

In [ ]:
import re

POLLUTANTS = ["pm25", "pm10", "no", "no2", "nox", "nh3", "so2", "co", "o3", "benzene", "toluene"]
MET = ["at", "rh", "ws", "wd", "rf", "sr"]   # air temp, humidity, wind speed/direction, rain, solar radiation
MEASURES = POLLUTANTS + MET

PARAM_MAP = {
    "PM2.5": "pm25", "PM10": "pm10", "NO": "no", "NO2": "no2", "NOx": "nox", "NH3": "nh3",
    "SO2": "so2", "CO": "co", "Ozone": "o3", "Benzene": "benzene", "Toluene": "toluene",
    "AT": "at", "Temp": "temp", "RH": "rh", "WS": "ws", "WD": "wd", "RF": "rf", "SR": "sr",
}
CANONICAL_UNIT = {
    "pm25": "ug/m3", "pm10": "ug/m3", "no": "ug/m3", "no2": "ug/m3", "nox": "ppb",
    "nh3": "ug/m3", "so2": "ug/m3", "co": "mg/m3", "o3": "ug/m3", "benzene": "ug/m3",
    "toluene": "ug/m3",
}
CONVERSIONS = {                       # (pollutant, unit found in file): multiply by this
    ("no", "ppb"): 1.227, ("no", "ppm"): 1227.0, ("no", "mg/m3"): 1000.0,
    ("nox", "ug/m3"): 1 / 1.88, ("nox", "ppm"): 1000.0,
    ("nh3", "ppb"): 0.696, ("o3", "ppb"): 1.963,
    ("co", "mg/Nm3"): 1.0, ("co", "ug/m3"): 1e-3, ("co", "ng/m3"): 1e-6,
    ("benzene", "mg/m3"): 1000.0,
}

`map_column` handles one header token. The regular expression splits `"PM2.5 (ug/m3)"` into a parameter part (`PM2.5`) and a unit part (`ug/m3`):

- `(?P<param>.+?)` captures the name (the `?` makes it stop as early as possible),
- `\s*\(` skips spaces and the opening bracket,
- `(?P<unit>[^)]*)` captures everything up to the closing bracket.

It returns `(standard_name, factor, note)`, or `None` if the column should be dropped (for example the date columns, or a unit we cannot convert safely).

In [ ]:
HEADER_RE = re.compile(r"^(?P<param>.+?)\s*\((?P<unit>[^)]*)\)$")

def map_column(token):
    m = HEADER_RE.match(token.strip())
    if not m:                                   # e.g. "From Date", or a duplicate like "RF (mm).1"
        return None
    param, unit = m.group("param").strip(), m.group("unit").strip()
    canon = PARAM_MAP.get(param)
    if canon is None:                           # a sensor we do not use
        return None
    if canon not in CANONICAL_UNIT:             # weather columns: keep as they are, range-check later
        return canon, 1.0, "met"
    if unit == CANONICAL_UNIT[canon]:
        return canon, 1.0, "canonical"
    if unit == "":
        return canon, 1.0, "unit missing - assumed canonical"
    if (canon, unit) in CONVERSIONS:
        return canon, CONVERSIONS[(canon, unit)], f"converted from {unit}"
    return None

# quick test on a few real header tokens
for t in ["PM2.5 (ug/m3)", "CO (ug/m3)", "Ozone (ppb)", "From Date", "Temp (degree C)"]:
    print(f"{t:18} -> {map_column(t)}")

### 4.2 Reading one layout group

For each group of files with the same header we:

- build a schema of plain string columns `c0, c1, c2, ...` (one per position). Reading everything as text first means a stray bad value cannot crash the read,
- read the files with `header=False` and then throw away the header line with a filter (this avoids hundreds of Spark warnings),
- turn each useful column into a number with `try_cast("double")` (bad text becomes null instead of an error) and multiply by its unit factor,
- get the station ID from the file name. Spark gives every row a hidden `_metadata.file_name` column, and the regex `^([A-Z]+\d+)\.csv$` keeps just `DL001`,
- parse `From Date` into a timestamp. We use `timestamp_ntz` ("no time zone") so Spark treats it as wall-clock Indian time and never shifts it.

Some stations have both an `AT` and a `Temp` column for air temperature, so `coalesce` takes whichever one is not null. Any standard column that a group does not have is filled with null, so every group ends up with exactly the same columns.

In [ ]:
mapping_rows = []      # we record what happened to every raw column, for the report

def build_group_df(header, files):
    schema = StructType([StructField(f"c{i}", StringType()) for i in range(len(header))])
    df = (spark.read.csv(files, schema=schema, header=False, mode="PERMISSIVE")
          .filter(F.col("c0") != "From Date"))           # drop the header line of each file

    exprs = {}
    for i, token in enumerate(header):
        mapped = map_column(token)
        mapping_rows.append({"raw_column": token.strip(), "files": len(files),
                             "canonical": mapped[0] if mapped else None,
                             "factor": mapped[1] if mapped else None,
                             "note": mapped[2] if mapped else "dropped"})
        if mapped and mapped[0] not in exprs:            # if a name appears twice, the first one wins
            exprs[mapped[0]] = F.col(f"c{i}").try_cast("double") * F.lit(mapped[1])

    at, temp = exprs.pop("at", None), exprs.pop("temp", None)
    if at is not None or temp is not None:
        exprs["at"] = F.coalesce(*[e for e in (at, temp) if e is not None])

    cols = [
        F.regexp_extract(F.col("_metadata.file_name"), r"^([A-Z]+\d+)\.csv$", 1).alias("station_id"),
        F.try_to_timestamp(F.col("c0"), F.lit("yyyy-MM-dd HH:mm:ss")).cast("timestamp_ntz").alias("ts"),
    ]
    cols += [(exprs[m] if m in exprs else F.lit(None).cast("double")).alias(m) for m in MEASURES]
    return df.select(*cols)

### 4.3 Stack all groups and write Bronze

`unionByName` stacks two DataFrames by matching column names (not positions). `reduce` from `functools` applies it again and again: `((g1 ∪ g2) ∪ g3) ∪ ...`, until all 198 groups are one DataFrame.

Nothing has actually been read yet. Spark is **lazy**: so far it has only written down a plan. The `.write` is an *action*, and that is the moment Spark reads all the CSVs in parallel and writes the result. `repartition(48)` splits the output into 48 evenly sized Parquet files.

Parquet stores data column by column and compresses it, so it is much smaller than CSV and much faster to query.

In [ ]:
from functools import reduce

BRONZE_DIR = f"{LAKE_DIR}/bronze_hourly"
STATIONS_DIR = f"{LAKE_DIR}/silver_stations"

dfs = [build_group_df(h, f) for h, f in groups.items()]
hourly = reduce(lambda a, b: a.unionByName(b), dfs)

with timed("CSV -> Bronze Parquet"):
    hourly.repartition(48).write.mode("overwrite").parquet(BRONZE_DIR)

# the station metadata: which state and city each station belongs to
stations = spark.read.csv(STATIONS_CSV, header=True, inferSchema=True)
stations = stations.select(F.trim("file_name").alias("station_id"),
                           *[F.trim(c).alias(c) for c in ("state", "city", "agency", "station_location")],
                           "start_year", "start_month_num")
stations.write.mode("overwrite").parquet(STATIONS_DIR)

Let's check what we got: the row count, the schema, and how much disk space Parquet saved. `du -sb` gives a folder's size in bytes.

In [ ]:
def folder_bytes(path):
    return int(subprocess.run(["du", "-sb", path], capture_output=True, text=True).stdout.split()[0])

bronze = spark.read.parquet(BRONZE_DIR)
print(f"hourly rows: {bronze.count():,}")
print(f"raw CSV: {raw_bytes / 1e9:.2f} GB  ->  Bronze Parquet: {folder_bytes(BRONZE_DIR) / 1e9:.2f} GB")
bronze.agg(F.min("ts"), F.max("ts"), F.sum(F.col("ts").isNull().cast("int")).alias("unparsed_ts")).show()
bronze.printSchema()

Full run: **14,295,663 hourly rows**, every timestamp parsed, 1.7 GB of CSV shrunk to under 0.5 GB of Parquet.

And here is what the mapping did to the raw column names, grouped and sorted by how many files used each one. The `converted from ...` rows are the unit fixes.

In [ ]:
import pandas as pd

mapping = (pd.DataFrame(mapping_rows)
           .groupby(["raw_column", "canonical", "factor", "note"], dropna=False)["files"].sum()
           .reset_index().sort_values("files", ascending=False))
print(len(mapping), "distinct raw column names")
display(mapping[mapping.note.str.startswith("converted")])
display(mapping.head(10))

## 5. Clean: Bronze to Silver

Our main rule: **null the bad value, keep the good ones.** If the CO sensor fails for an hour, that hour's PM2.5 reading is still fine, so we only remove a whole row when nothing in it is usable.

| Rule | What it does |
|---|---|
| R1 | drop duplicate (station, hour) rows |
| R2 | drop placeholder rows where every reading is empty |
| R3 | null zero or negative pollutant values (a concentration cannot be ≤ 0) |
| R4 | null values outside physically possible ranges |
| R5 | null PM "stuck sensor" runs: the exact same value for 24+ hours in a row |
| R6 | null PM2.5 and PM10 when PM2.5 > PM10 (PM2.5 is part of PM10, so this is impossible) |
| R7 | drop rows that have no pollutant value left |

Every rule writes a line into a log, saying how many rows it removed or how many values it nulled.

In [ ]:
UPPER = {"pm25": 1000, "pm10": 2000, "no": 1000, "no2": 1000, "nox": 1000, "nh3": 1000,
         "so2": 1000, "co": 50, "o3": 1000, "benzene": 500, "toluene": 1000}
MET_RANGE = {"at": (-10, 55), "rh": (1, 100), "ws": (0, 40), "wd": (0, 360),
             "rf": (0, 500), "sr": (0, 1500)}
STUCK_HOURS = 24

def nonnull_counts(df, cols):
    # F.count(col) only counts non-null values, so this gives "how many valid readings per column"
    return df.agg(*[F.count(c).alias(c) for c in cols]).first().asDict()

### 5.1 R1 and R2: duplicates and empty rows

`dropDuplicates([...])` keeps one row per station and hour. `coalesce(a, b, c, ...)` returns the first non-null value, so if it returns null, *every* reading in that row was empty. Those are placeholder rows from before a station actually started working.

`.cache()` tells Spark to keep this DataFrame in memory after the first time it is computed, because we are about to use it several times.

In [ ]:
log = []
n0 = bronze.count()
log.append({"rule": "R0 raw hourly rows", "rows_after": n0, "rows_removed": 0, "values_nulled": 0})

df = bronze.filter(F.col("ts").isNotNull()).dropDuplicates(["station_id", "ts"])
n1 = df.count()
log.append({"rule": "R1 duplicate (station, hour) rows", "rows_after": n1, "rows_removed": n0 - n1, "values_nulled": 0})

df = df.filter(F.coalesce(*[F.col(c) for c in MEASURES]).isNotNull()).cache()
n2 = df.count()
log.append({"rule": "R2 empty placeholder rows", "rows_after": n2, "rows_removed": n1 - n2, "values_nulled": 0})
raw_nonnull = nonnull_counts(df, MEASURES)
print(f"{n0:,} -> {n1:,} -> {n2:,} rows")

### 5.2 R3 and R4: impossible values

These rules do not remove rows, they null single values. `F.when(condition, value)` is Spark's if/else: when the condition is false and there is no `.otherwise(...)`, the result is null. So `F.when(F.col(c) > 0, F.col(c))` means "keep the value if it is positive, otherwise null".

`value_rule` compares the valid-value counts before and after a rule, so we know how many values that rule nulled in each column.

In [ ]:
def value_rule(name, new_df, prev_counts):
    counts = nonnull_counts(new_df, MEASURES)
    nulled = {c: prev_counts[c] - counts[c] for c in MEASURES}
    log.append({"rule": name, "rows_after": n2, "rows_removed": 0, "values_nulled": sum(nulled.values()),
                **{f"nulled_{c}": v for c, v in nulled.items() if v}})
    return counts

# R3: pollutants must be > 0 (weather columns pass through untouched)
df3 = df.select("station_id", "ts", *[
    F.when(F.col(c) > 0, F.col(c)).alias(c) if c in POLLUTANTS else F.col(c) for c in MEASURES])
c3 = value_rule("R3 non-positive pollutant values", df3, raw_nonnull)

# R4: everything must sit inside a physically possible range
def in_range(c):
    lo, hi = (0, UPPER[c]) if c in UPPER else MET_RANGE[c]
    return F.when(F.col(c).between(lo, hi), F.col(c)).alias(c)

df4 = df3.select("station_id", "ts", *[in_range(c) for c in MEASURES]).cache()
c4 = value_rule("R4 out-of-range values", df4, c3)

### 5.3 R5: stuck sensors, using window functions

Real air never stays at *exactly* the same PM value for a whole day, so a run of 24+ identical hourly readings means the instrument froze. Finding those runs needs a **window function**, which looks at neighbouring rows without collapsing them like `GROUP BY` would.

How it works, step by step, for one station ordered by time:
1. `F.lag(col)` gives the previous hour's value.
2. `_chg` is 1 whenever the value changed from the previous hour (or either is null), else 0.
3. A running sum of `_chg` (`F.sum(...).over(w)`) gives a **run number**: it only goes up when the value changes, so all rows in one unchanged stretch share the same number.
4. Counting rows per (station, run number) gives each run's length. Runs of 24 or more are nulled.

For example, values `50, 50, 50, 61` give `_chg = 1, 0, 0, 1` and run numbers `1, 1, 1, 2`.

In [ ]:
def null_stuck_runs(df, col):
    w = Window.partitionBy("station_id").orderBy("ts")          # one ordered timeline per station
    prev = F.lag(col).over(w)
    changed = F.when(F.col(col).isNull() | prev.isNull() | (F.col(col) != prev), 1).otherwise(0)
    df = df.withColumn("_chg", changed).withColumn("_run", F.sum("_chg").over(w))
    run_len = F.count(F.lit(1)).over(Window.partitionBy("station_id", "_run"))
    df = df.withColumn(col, F.when(F.col(col).isNotNull() & (run_len >= STUCK_HOURS), None)
                            .otherwise(F.col(col)))
    return df.drop("_chg", "_run")

with timed("stuck-sensor check"):
    df5 = null_stuck_runs(null_stuck_runs(df4, "pm25"), "pm10").cache()
    c5 = value_rule(f"R5 stuck-sensor runs (>= {STUCK_HOURS} h constant PM)", df5, c4)
df4.unpersist()     # free the memory, we do not need df4 any more

### 5.4 R6 and R7: inconsistent PM pairs and empty rows

If PM2.5 is bigger than PM10 in the same hour, at least one of the two sensors is wrong, and we cannot tell which, so both are nulled. `withColumn` replaces a column (or adds a new one).

In [ ]:
bad_pair = F.col("pm25") > F.col("pm10")
df6 = (df5.withColumn("_bad", bad_pair)
          .withColumn("pm25", F.when(F.col("_bad"), None).otherwise(F.col("pm25")))
          .withColumn("pm10", F.when(F.col("_bad"), None).otherwise(F.col("pm10")))
          .drop("_bad"))
value_rule("R6 PM2.5 > PM10 inconsistent pairs", df6, c5)

df7 = df6.filter(F.coalesce(*[F.col(c) for c in POLLUTANTS]).isNotNull())   # R7

### 5.5 Add location and time columns, then write Silver

- We join in the state and city from the station table. That table has only 453 rows, so `F.broadcast` sends a full copy to every worker. That way the 12 million hourly rows never have to be shuffled around for the join. This is called a **broadcast join**.
- We add `date`, `year`, `month`, `hour`, day of week and `season` (India's four seasons as CPCB uses them).
- `partitionBy("year")` writes one sub-folder per year (`year=2019/`, `year=2020/`, ...). Later, a query that asks only about 2022 opens only that folder. This is **partition pruning**.

In [ ]:
SILVER_DIR = f"{LAKE_DIR}/silver_hourly"

def season_expr(month):
    return (F.when(month.isin(12, 1, 2), "Winter")
             .when(month.isin(3, 4, 5), "Summer")
             .when(month.isin(6, 7, 8, 9), "Monsoon")
             .otherwise("Post-monsoon"))

st = spark.read.parquet(STATIONS_DIR).select("station_id", "state", "city")
silver = (df7.join(F.broadcast(st), "station_id", "left")
             .withColumn("date", F.to_date("ts"))
             .withColumn("year", F.year("ts"))
             .withColumn("month", F.month("ts"))
             .withColumn("hour", F.hour("ts"))
             .withColumn("dow", F.dayofweek("ts"))          # 1 = Sunday ... 7 = Saturday
             .withColumn("season", season_expr(F.month("ts"))))

with timed("write Silver (partitioned by year)"):
    silver.repartition("year").write.mode("overwrite").partitionBy("year").parquet(SILVER_DIR)
df.unpersist(); df5.unpersist()

silver = spark.read.parquet(SILVER_DIR)
n7 = silver.count()
log.append({"rule": "R7 rows with no valid pollutant left", "rows_after": n7, "rows_removed": n2 - n7, "values_nulled": 0})
!ls {SILVER_DIR} | head -5

### 5.6 The cleaning log

This table is what the cleaning chart on slide 7 is built from. On the full data: about 2.2 million empty placeholder rows were dropped, about 2.5 million zero/negative values were nulled (mostly benzene and toluene sensors reporting 0), and about 158 thousand values were outside possible ranges.

In [ ]:
cleaning_log = pd.DataFrame(log).fillna(0)
display(cleaning_log[["rule", "rows_after", "rows_removed", "values_nulled"]])

final_nonnull = nonnull_counts(silver, MEASURES)
completeness = pd.DataFrame([{"measure": c,
                              "pct_available_raw": round(100 * raw_nonnull[c] / n2, 1),
                              "pct_available_clean": round(100 * final_nonnull[c] / n7, 1)} for c in MEASURES])
display(completeness)

## 6. Daily AQI: Silver to Gold

India's National Air Quality Index (CPCB, 2014) works like this:

1. Each pollutant gets a daily concentration. For PM2.5, PM10, NO2, SO2 and NH3 that is the 24-hour average, and it only counts if at least 16 hours have data. For CO and ozone it is the **highest 8-hour average** of the day.
2. That concentration is turned into a **sub-index** from 0 to 500 using CPCB's breakpoint table, by drawing a straight line inside the band it falls in.
3. The day's AQI is the **worst** (maximum) sub-index. It is only reported if at least 3 pollutants are available and one of them is PM2.5 or PM10.

### 6.1 The breakpoint table and the sub-index formula

`BREAKPOINTS` holds the lower edge of each of the six bands for each pollutant. Inside a band, the formula is just a straight line:

`sub_index = (I_hi - I_lo) / (B_hi - B_lo) * (C - B_lo) + I_lo`

where C is the concentration, B is the band's concentration range and I is the band's AQI range. `sub_index` builds one big `when ... when ... when` chain, one branch per band. The last band is open-ended, so we use the extrapolation span that CPCB's calculator uses, and cap the result at 500.

In [ ]:
AQI_BANDS = [(0, 50), (51, 100), (101, 200), (201, 300), (301, 400), (401, 500)]
BREAKPOINTS = {   # pollutant: (lower edge of each band, span used above the top band)
    "pm25": ([0, 31, 61, 91, 121, 251], 130),
    "pm10": ([0, 51, 101, 251, 351, 431], 80),
    "no2":  ([0, 41, 81, 181, 281, 401], 120),
    "so2":  ([0, 41, 81, 381, 801, 1601], 800),
    "nh3":  ([0, 201, 401, 801, 1201, 1801], 600),
    "co8":  ([0, 1.1, 2.1, 10.1, 17.1, 34.1], 17),
    "o38":  ([0, 51, 101, 169, 209, 749], 120),
}
CATEGORIES = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
MIN_HOURS_24H, MIN_HOURS_8H = 16, 6

def sub_index(col, name):
    lows, last_span = BREAKPOINTS[name]
    c = F.col(col)
    expr = None
    for i in range(len(lows)):
        b_lo = lows[i]
        b_hi = lows[i + 1] - (0.1 if name == "co8" else 1) if i + 1 < len(lows) else b_lo + last_span
        i_lo, i_hi = AQI_BANDS[i]
        val = (i_hi - i_lo) / (b_hi - b_lo) * (c - b_lo) + i_lo
        cond = c < lows[i + 1] if i + 1 < len(lows) else F.lit(True)
        expr = F.when(cond, val) if expr is None else expr.when(cond, val)
    return F.when(c.isNull(), None).otherwise(F.least(expr, F.lit(500.0)))

A quick sanity check by hand. PM2.5 = 100 falls in the band 91-120, which maps to AQI 201-300:
(300 - 201) / (120 - 91) × (100 - 91) + 201 = 99/29 × 9 + 201 ≈ **231.7**, which is "Poor". Spark should agree:

In [ ]:
test = spark.createDataFrame([(25.0,), (100.0,), (300.0,)], ["pm25"])
test.select("pm25", F.round(sub_index("pm25", "pm25"), 1).alias("pm25_sub_index")).show()

### 6.2 The 8-hour rolling averages for CO and ozone

This is another window function, but this time the window is defined by **time**, not by number of rows. We convert each timestamp to seconds since 1970 (`unix_seconds`) and use `rangeBetween(-7 * 3600, 0)`, meaning "every row from 7 hours before this one up to this one", which is 8 clock hours. If an hour is missing, the window does not quietly stretch back further, which a row-based window would do. The average only counts if at least 6 of those 8 hours have data.

In [ ]:
hourly = spark.read.parquet(SILVER_DIR)

epoch = F.unix_seconds(F.col("ts").cast("timestamp"))
w8 = Window.partitionBy("station_id").orderBy("_epoch").rangeBetween(-7 * 3600, 0)
hourly = (hourly.withColumn("_epoch", epoch)
          .withColumn("co8", F.when(F.count("co").over(w8) >= MIN_HOURS_8H, F.avg("co").over(w8)))
          .withColumn("o38", F.when(F.count("o3").over(w8) >= MIN_HOURS_8H, F.avg("o3").over(w8))))

### 6.3 Daily averages, sub-indices and the final AQI

- `groupBy(...).agg(...)` collapses 24 hourly rows into one row per station per day. `mean24` returns the average only when 16+ hours had data.
- We add one sub-index column per pollutant, `si_pm25`, `si_pm10`, ...
- `n_avail` counts how many sub-indices exist for that day. Python's `sum()` works here because adding Spark columns gives another Spark column.
- `F.greatest(...)` picks the largest sub-index, which is the AQI.
- The **prominent pollutant** is whichever pollutant gave that maximum.

In [ ]:
GOLD_DAILY_DIR = f"{LAKE_DIR}/gold_daily"

def mean24(c):
    return F.when(F.count(c) >= MIN_HOURS_24H, F.avg(c)).alias(c)

keys = ["station_id", "state", "city", "date", "year", "month", "season"]
daily = hourly.groupBy(*keys).agg(
    *[mean24(c) for c in MEASURES],
    F.max("co8").alias("co8"), F.max("o38").alias("o38"),       # highest 8-hour average of the day
    F.count("*").alias("hours_reported"))

daily = daily.select("*", *[sub_index(n, n).alias(f"si_{n}") for n in BREAKPOINTS])
si_cols = [F.col(f"si_{n}") for n in BREAKPOINTS]
n_avail = sum(F.when(c.isNotNull(), 1).otherwise(0) for c in si_cols)
has_pm = F.col("si_pm25").isNotNull() | F.col("si_pm10").isNotNull()
daily = daily.withColumn("aqi", F.when((n_avail >= 3) & has_pm, F.round(F.greatest(*si_cols))))

cat = None
for (lo, hi), name in zip(AQI_BANDS, CATEGORIES):
    cond = F.col("aqi") <= hi
    cat = F.when(cond, name) if cat is None else cat.when(cond, name)
prominent = F.when(F.col("aqi").isNotNull(), F.coalesce(*[
    F.when(F.col(f"si_{n}") == F.greatest(*si_cols), F.lit(n)) for n in BREAKPOINTS]))
daily = daily.withColumn("aqi_category", cat).withColumn("prominent_pollutant", prominent)

with timed("Silver -> Gold daily AQI"):
    daily.write.mode("overwrite").partitionBy("year").parquet(GOLD_DAILY_DIR)

gold = spark.read.parquet(GOLD_DAILY_DIR)
gold.agg(F.count("*").alias("station_days"), F.count("aqi").alias("days_with_aqi"),
         F.round(F.avg("aqi"), 1).alias("mean_aqi")).show()

One Delhi day just after Diwali 2021, to see the Gold layer. Notice how PM2.5 decides the AQI on almost every row.

In [ ]:
(gold.filter("city = 'Delhi' AND date = '2021-11-05'")
     .select("station_id", "pm25", "pm10", "no2", "co8", "o38", "aqi", "aqi_category", "prominent_pollutant")
     .show(8))

## 7. Spark SQL: the five research questions

Now that the data is clean, we can ask questions in plain SQL. `createOrReplaceTempView` gives a DataFrame a table name that SQL can use. Spark turns each SQL query into exactly the same kind of plan as the DataFrame code above, so there is no speed difference, it is just easier to read.

We `.cache()` the daily table because about 15 queries below all read it.

Only the small final answers come back to Python with `.toPandas()`. The heavy work (scanning millions of rows) always stays in Spark.

In [ ]:
spark.read.parquet(SILVER_DIR).createOrReplaceTempView("hourly")
spark.read.parquet(GOLD_DAILY_DIR).cache().createOrReplaceTempView("daily")
spark.read.parquet(STATIONS_DIR).createOrReplaceTempView("stations")

METROS = ["Delhi", "Mumbai", "Kolkata", "Chennai", "Bengaluru", "Hyderabad"]
IGP_STATES = ["Punjab", "Haryana", "Delhi", "Chandigarh", "Uttar Pradesh", "Bihar", "West Bengal", "Jharkhand"]
igp = ", ".join(f"'{s}'" for s in IGP_STATES)       # -> 'Punjab', 'Haryana', ...  (ready to paste into SQL)
metros = ", ".join(f"'{c}'" for c in METROS)

# same daily table, plus a column saying whether the state is in the Indo-Gangetic Plain
spark.sql(f'''CREATE OR REPLACE TEMP VIEW daily_r AS
              SELECT *, CASE WHEN state IN ({igp}) THEN 'Indo-Gangetic Plain'
                             ELSE 'Rest of India' END AS region
              FROM daily''')

def sql(query):
    return spark.sql(query).toPandas()

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

### Data overview: the network keeps growing

Stations per year. This is the "Velocity" point on the 5V slide: the network went from 29 stations in 2010 to 434 in 2023.

In [ ]:
coverage = sql('''SELECT year, COUNT(DISTINCT station_id) AS stations, COUNT(*) AS hourly_rows
                  FROM hourly GROUP BY year ORDER BY year''')
ax = coverage.plot.bar(x="year", y="stations", legend=False, color="#4C72B0")
ax.set_ylabel("active stations"); ax.set_title("Monitoring stations reporting each year"); plt.show()

### RQ1: Where is the air worst?

City ranking for 2019-2022. To keep the ranking fair we only include cities with enough data (`HAVING` filters *after* grouping, `WHERE` filters *before*): at least 600 days of AQI and 300 days of PM2.5.

`AVG(CASE WHEN aqi > 200 THEN 1.0 ELSE 0 END)` is a neat trick: the average of a bunch of 1s and 0s is the share of days that were "Poor" or worse.

In [ ]:
city_rank = sql('''
    SELECT city, state, COUNT(DISTINCT station_id) AS stations,
           ROUND(AVG(pm25), 1) AS mean_pm25, ROUND(AVG(aqi), 0) AS mean_aqi,
           ROUND(100 * AVG(CASE WHEN aqi > 200 THEN 1.0 ELSE 0 END), 1) AS pct_days_poor_or_worse
    FROM daily
    WHERE year BETWEEN 2019 AND 2022 AND aqi IS NOT NULL
    GROUP BY city, state
    HAVING COUNT(DISTINCT date) >= 600 AND COUNT(pm25) >= 300
    ORDER BY mean_pm25 DESC''')
print(f"{(city_rank.mean_pm25 > 40).sum()} of {len(city_rank)} cities are above India's annual PM2.5 limit of 40")
display(city_rank.head(15))

top = city_rank.head(15).iloc[::-1]
ax = top.plot.barh(x="city", y="mean_pm25", legend=False, color="#C44E52")
ax.axvline(40, color="k", ls="--", lw=1); ax.text(41, 0, "national limit (40)", fontsize=8)
ax.set_xlabel("mean PM2.5, ug/m3 (2019-2022)"); ax.set_ylabel(""); plt.show()

North India against the rest, month by month. The gap opens up from October to January: in November the Indo-Gangetic Plain averages about 2.7 times the PM2.5 of the rest of the country. The plain is a basin with the Himalayas along one side, winter inversions trap the air near the ground, and crop burning in Punjab and Haryana adds smoke in October-November.

In [ ]:
region_monthly = sql('''
    SELECT region, month, ROUND(AVG(pm25), 1) AS mean_pm25
    FROM daily_r WHERE year BETWEEN 2019 AND 2022
    GROUP BY region, month ORDER BY region, month''')
ax = region_monthly.pivot(index="month", columns="region", values="mean_pm25").plot(marker="o")
ax.set_ylabel("mean PM2.5, ug/m3"); ax.set_xticks(range(1, 13)); ax.set_title("PM2.5 by month, 2019-2022"); plt.show()

display(sql('''SELECT region, prominent_pollutant, COUNT(*) AS days
               FROM daily_r WHERE year BETWEEN 2019 AND 2022 AND aqi > 100
               GROUP BY region, prominent_pollutant ORDER BY region, days DESC'''))

### RQ2: When is it worst? Season, hour of day, and the long-term trend

Average PM2.5 by hour of the day, one line per season. Winter nights are the worst: after sunset the ground cools, the air near it stops mixing upward, and pollution builds up under that "lid".

In [ ]:
diurnal = sql('''SELECT season, hour, ROUND(AVG(pm25), 1) AS mean_pm25
                 FROM hourly WHERE year BETWEEN 2019 AND 2022
                 GROUP BY season, hour ORDER BY season, hour''')
ax = diurnal.pivot(index="hour", columns="season", values="mean_pm25").plot()
ax.set_ylabel("mean PM2.5, ug/m3"); ax.set_xticks(range(0, 24, 3)); ax.set_title("PM2.5 by hour of day and season"); plt.show()

Delhi as a month × hour heatmap. `pivot` turns the long table (month, hour, value) into a grid with months as rows and hours as columns. The darkest cells are November to January, roughly 8 pm to 1 am.

In [ ]:
delhi = sql('''SELECT month, hour, ROUND(AVG(pm25), 1) AS mean_pm25
               FROM hourly WHERE city = 'Delhi' AND year BETWEEN 2018 AND 2022
               GROUP BY month, hour''')
grid = delhi.pivot(index="month", columns="hour", values="mean_pm25")
plt.figure(figsize=(10, 4))
plt.imshow(grid, aspect="auto", cmap="magma_r")
plt.colorbar(label="PM2.5, ug/m3"); plt.yticks(range(12), grid.index); plt.xlabel("hour"); plt.ylabel("month")
plt.title("Delhi PM2.5, 2018-2022"); plt.grid(False); plt.show()

**The trend, done fairly.** A plain national average drops from 131 in 2010 to 60 in 2022, but most of that is just new, cleaner stations in the south joining the network. So we compare **the same stations every year**: only stations with at least 200 valid PM2.5 days in *each* of 2018-2022.

This query uses `WITH`, which defines temporary named results (`per_year`, then `panel`) that the final `SELECT` can use, a bit like variables in SQL. `HAVING COUNT(*) = 5` keeps stations that passed in all five years. `JOIN ... USING (station_id)` keeps only those stations.

In [ ]:
panel = sql('''
    WITH per_year AS (
        SELECT station_id, year, AVG(pm25) AS pm25, COUNT(pm25) AS n
        FROM daily WHERE year BETWEEN 2018 AND 2022 GROUP BY station_id, year),
    panel AS (
        SELECT station_id FROM per_year WHERE n >= 200
        GROUP BY station_id HAVING COUNT(*) = 5)
    SELECT year, COUNT(*) AS stations, ROUND(AVG(pm25), 1) AS mean_pm25,
           ROUND(PERCENTILE_APPROX(pm25, 0.5), 1) AS median_pm25
    FROM per_year JOIN panel USING (station_id)
    GROUP BY year ORDER BY year''')
display(panel)
if len(panel) == 5:
    print(f"change 2018 -> 2022 at the same {panel.stations[0]} stations: "
          f"{100 * (panel.mean_pm25.iloc[-1] / panel.mean_pm25.iloc[0] - 1):.0f}%")

### RQ3: How big is the Diwali effect?

Diwali moves every year, so we make a tiny table of Diwali dates and join the daily data to it. The join condition is a **range**: every day from 10 days before to 10 days after that year's Diwali. `DATEDIFF` then gives each day's offset from Diwali (-10 ... +10), so we can line up all the years on top of each other. We start in 2015 because Delhi had only one or two PM2.5 monitors before that.

In [ ]:
DIWALI = ["2015-11-11", "2016-10-30", "2017-10-19", "2018-11-07", "2019-10-27",
          "2020-11-14", "2021-11-04", "2022-10-24"]
(spark.createDataFrame([(d,) for d in DIWALI], ["diwali"])
      .selectExpr("CAST(diwali AS DATE) AS diwali", "YEAR(CAST(diwali AS DATE)) AS dyear")
      .createOrReplaceTempView("diwali"))

offsets = sql('''
    SELECT CASE WHEN d.city = 'Delhi' THEN 'Delhi' ELSE 'Rest of India' END AS area,
           DATEDIFF(d.date, w.diwali) AS day_offset, ROUND(AVG(d.pm25), 1) AS mean_pm25
    FROM daily d JOIN diwali w
      ON d.date BETWEEN DATE_SUB(w.diwali, 10) AND DATE_ADD(w.diwali, 10)
    GROUP BY 1, 2 ORDER BY 1, 2''')
ax = offsets.pivot(index="day_offset", columns="area", values="mean_pm25").plot(marker="o")
ax.axvline(0, color="k", ls="--", lw=1); ax.set_xlabel("days from Diwali"); ax.set_ylabel("mean PM2.5, ug/m3")
ax.set_title("PM2.5 around Diwali, 2015-2022 average"); plt.show()

Year by year for Delhi: the day after Diwali compared with a "normal" baseline (days -10 to -4, before the festival smoke). The trick `MAX(CASE WHEN off = 1 THEN pm25 END)` picks out the single day-after value inside a `GROUP BY`. 2020 is the odd one out: the baseline was already 360 because of crop-burning smoke, so the ratio is below 1.

In [ ]:
display(sql('''
    WITH city_day AS (
        SELECT w.dyear, DATEDIFF(d.date, w.diwali) AS off, AVG(d.pm25) AS pm25
        FROM daily d JOIN diwali w
          ON d.date BETWEEN DATE_SUB(w.diwali, 10) AND DATE_ADD(w.diwali, 10)
        WHERE d.city = 'Delhi' GROUP BY w.dyear, DATEDIFF(d.date, w.diwali))
    SELECT dyear AS year,
           ROUND(AVG(CASE WHEN off BETWEEN -10 AND -4 THEN pm25 END), 1) AS baseline_pm25,
           ROUND(MAX(CASE WHEN off = 1 THEN pm25 END), 1) AS day_after_pm25,
           ROUND(MAX(CASE WHEN off = 1 THEN pm25 END)
                 / AVG(CASE WHEN off BETWEEN -10 AND -4 THEN pm25 END), 2) AS ratio
    FROM city_day GROUP BY dyear ORDER BY dyear'''))

The hourly picture shows how sharp the spike really is. Daily averages hide it: in Delhi the hourly mean peaks at around 600 ug/m3 close to midnight on Diwali night, which points to fireworks rather than crop burning (crop smoke would not follow the clock like that).

In [ ]:
diwali_hourly = sql('''
    SELECT CAST((UNIX_SECONDS(CAST(h.ts AS TIMESTAMP))
                 - UNIX_SECONDS(CAST(w.diwali AS TIMESTAMP))) / 3600 AS INT) AS hour_offset,
           ROUND(AVG(h.pm25), 1) AS mean_pm25
    FROM hourly h JOIN diwali w
      ON h.ts >= CAST(DATE_SUB(w.diwali, 1) AS TIMESTAMP_NTZ)
     AND h.ts <  CAST(DATE_ADD(w.diwali, 3) AS TIMESTAMP_NTZ)
    WHERE h.city = 'Delhi' AND w.dyear >= 2017
    GROUP BY 1 ORDER BY 1''')
ax = diwali_hourly.plot(x="hour_offset", y="mean_pm25", legend=False, color="#8172B3")
ax.axvline(0, color="k", ls="--", lw=1); ax.axvline(24, color="k", ls=":", lw=1)
ax.set_xlabel("hours from the start of Diwali day"); ax.set_ylabel("PM2.5, ug/m3"); ax.set_title("Delhi, hourly, 2017-2022"); plt.show()
print("peak:", diwali_hourly.mean_pm25.max(), "ug/m3 at hour", diwali_hourly.loc[diwali_hourly.mean_pm25.idxmax(), "hour_offset"])

### RQ4: What did the 2020 lockdown do?

The lockdown is a natural experiment: traffic and industry were switched off almost overnight. We compare the lockdown window (25 March to 31 May) in 2020 with **the same window in 2019, at the same station**. `DATE_FORMAT(date, 'MM-dd')` turns a date into text like `04-15`, so we can filter the same calendar window in any year.

The **self-join** (`lockdown a JOIN lockdown b`) puts each station's 2019 row next to its 2020 row. We then take the median percentage change across stations, so one odd station cannot drag the result. Only stations with 50+ days of data in both years count.

In [ ]:
spark.sql('''CREATE OR REPLACE TEMP VIEW lockdown AS
    SELECT station_id, city, year,
           AVG(pm25) AS pm25, AVG(pm10) AS pm10, AVG(no2) AS no2, AVG(co) AS co,
           AVG(so2) AS so2, AVG(o3) AS o3, COUNT(*) AS days
    FROM daily
    WHERE year IN (2019, 2020) AND DATE_FORMAT(date, 'MM-dd') BETWEEN '03-25' AND '05-31'
    GROUP BY station_id, city, year''')

pols = ["pm25", "pm10", "no2", "co", "so2", "o3"]
pct = ",\n".join(f"ROUND(100 * PERCENTILE_APPROX((b.{p} - a.{p}) / a.{p}, 0.5), 1) AS {p}" for p in pols)
change = sql(f'''
    SELECT COUNT(*) AS stations, {pct}
    FROM lockdown a JOIN lockdown b USING (station_id)
    WHERE a.year = 2019 AND b.year = 2020 AND a.days >= 50 AND b.days >= 50''')
display(change)

ax = change[pols].T.plot.bar(legend=False, color="#55A868")
ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("median % change vs 2019")
ax.set_title(f"Lockdown 2020 vs 2019, same {change.stations[0]} stations"); plt.show()

NO2 fell the most (about 45% on the full data) because it comes mostly from vehicles. Ozone barely moved, because ozone is not emitted directly, it forms from other gases in sunlight. The metros, side by side:

In [ ]:
display(sql(f'''
    SELECT a.city, COUNT(*) AS stations,
           ROUND(AVG(a.pm25), 1) AS pm25_2019, ROUND(AVG(b.pm25), 1) AS pm25_2020,
           ROUND(AVG(a.no2), 1) AS no2_2019, ROUND(AVG(b.no2), 1) AS no2_2020
    FROM lockdown a JOIN lockdown b USING (station_id)
    WHERE a.year = 2019 AND b.year = 2020 AND a.days >= 50 AND b.days >= 50 AND a.city IN ({metros})
    GROUP BY a.city ORDER BY a.city'''))

### RQ5: How does the weather relate to PM2.5?

`CORR(x, y)` is Spark's built-in Pearson correlation (-1 to +1). Temperature has the strongest link: colder days are more polluted. These are associations, not proof of cause.

In [ ]:
display(sql('''
    SELECT ROUND(CORR(pm25, ws), 3) AS pm25_vs_wind_speed,
           ROUND(CORR(pm25, at), 3) AS pm25_vs_temperature,
           ROUND(CORR(pm25, rh), 3) AS pm25_vs_humidity,
           ROUND(CORR(pm25, rf), 3) AS pm25_vs_rainfall,
           COUNT(*) AS station_days
    FROM daily WHERE year BETWEEN 2019 AND 2022'''))

Wind speed in bins of 0.5 m/s. `FLOOR(ws * 2) / 2` rounds down to the nearest 0.5 (e.g. 1.7 → 1.5), and `LEAST(..., 4.0)` puts everything above 4 m/s into one last bin. In winter, calm days are far worse than windy ones, because wind carries pollution away. In the monsoon, rain keeps PM low whatever the wind does.

In [ ]:
wind = sql('''
    SELECT season, CAST(LEAST(FLOOR(ws * 2) / 2, 4.0) AS DOUBLE) AS ws_bin, ROUND(AVG(pm25), 1) AS mean_pm25
    FROM daily WHERE year BETWEEN 2019 AND 2022 AND ws IS NOT NULL AND pm25 IS NOT NULL
    GROUP BY 1, 2 ORDER BY 1, 2''')
ax = wind.pivot(index="ws_bin", columns="season", values="mean_pm25").plot(marker="o")
ax.set_xlabel("daily mean wind speed, m/s"); ax.set_ylabel("mean PM2.5, ug/m3"); plt.show()

## 8. Spark RDDs: the low-level way

DataFrames are built on top of **RDDs** (Resilient Distributed Datasets), which are just distributed collections of Python objects. Here we redo a simple count, hourly rows and mean raw PM2.5 per state and year, straight from the raw CSV text, using only RDD operations. Then we check that it matches the DataFrame result exactly.

The pieces:
- `sc.textFile` reads text line by line. `sc.wholeTextFiles` reads whole files as `(path, content)` pairs, which we need because every file has its own header.
- `sc.broadcast` sends a small lookup table (station → state) to every worker once, instead of with every task.
- `flatMap` turns one input (a whole file) into many outputs (one per line). Our function uses `yield`, so it hands out results one at a time.
- Each output is a key-value pair: `((state, year), (1, is_valid, pm25))`.
- `reduceByKey(add)` adds up the values for each key, tuple by tuple. This is the "reduce" of MapReduce.
- `mapValues`, `join` and `sortBy` work like their names suggest.

All of these are **transformations** and they are lazy. Nothing runs until the `collect()` action at the end.

In [ ]:
# station -> state lookup, built from the raw metadata file
meta = sc.textFile(STATIONS_CSV)
head = meta.first()
state_map = (meta.filter(lambda l: l != head)
                 .map(lambda l: next(csv.reader([l])))           # proper CSV split (handles quotes)
                 .map(lambda r: (r[0].strip(), r[1].strip()))
                 .collectAsMap())
state_of = sc.broadcast(state_map)

def parse_station_file(record):
    path, content = record
    station = path.rsplit("/", 1)[-1].replace(".csv", "")
    state = state_of.value.get(station, "Unknown")
    lines = content.splitlines()
    header = lines[0].split(",")
    pm_idx = header.index("PM2.5 (ug/m3)") if "PM2.5 (ug/m3)" in header else None
    for line in lines[1:]:
        fields = line.split(",")
        year = fields[0][:4]
        pm = None
        if pm_idx is not None and pm_idx < len(fields) and fields[pm_idx]:
            try:
                pm = float(fields[pm_idx])
            except ValueError:
                pm = None
        valid = pm is not None and pm > 0
        yield (state, year), (1, 1 if valid else 0, pm if valid else 0.0)

def add(a, b):
    return a[0] + b[0], a[1] + b[1], a[2] + b[2]

In [ ]:
t0 = time.perf_counter()
files = sc.wholeTextFiles(",".join(station_files), minPartitions=24)
pairs = files.flatMap(parse_station_file)
totals = pairs.filter(lambda kv: "2010" <= kv[0][1] <= "2023").reduceByKey(add)
stats = totals.mapValues(lambda v: (v[0], v[1], round(v[2] / v[1], 2) if v[1] else None))

# a second pair RDD: number of stations per state, then join the two on state
stations_per_state = (sc.parallelize(list(state_map.items()))
                        .map(lambda kv: (kv[1], 1)).reduceByKey(lambda a, b: a + b))
joined = (stats.map(lambda kv: (kv[0][0], (kv[0][1],) + kv[1]))      # re-key by state only
               .join(stations_per_state)
               .map(lambda kv: (kv[0], kv[1][0][0], kv[1][1], *kv[1][0][1:]))
               .sortBy(lambda r: (r[0], r[1])))
rows = joined.collect()                                               # <- the action: everything runs now
print(f"RDD job: {time.perf_counter() - t0:.1f}s")

rdd_df = pd.DataFrame(rows, columns=["state", "year", "stations_in_state", "hourly_rows", "pm25_valid", "mean_pm25_raw"])
display(rdd_df.head(8))

**Lineage.** Spark remembers every step that built an RDD. If a worker dies and a partition is lost, Spark replays those steps for just that partition. That is the "Resilient" in RDD. `toDebugString` prints that chain; the indentation changes wherever data had to be shuffled.

In [ ]:
print(joined.toDebugString().decode()[:1500])

Cross-check against the DataFrame route on Bronze. On the full data both give **14,295,663 rows** with zero mismatched groups. MapReduce (next section) gives the same number too, which is how we know the pipeline is not losing rows.

In [ ]:
st_state = spark.read.parquet(STATIONS_DIR).select("station_id", "state")
df_counts = (spark.read.parquet(BRONZE_DIR).join(F.broadcast(st_state), "station_id")
             .groupBy("state", F.year("ts").cast("string").alias("year"))
             .agg(F.count("*").alias("hourly_rows_df"),
                  F.count(F.when(F.col("pm25") > 0, 1)).alias("pm25_valid_df"))
             .toPandas())
check = rdd_df.merge(df_counts, on=["state", "year"], how="outer")
mismatches = int(((check.hourly_rows != check.hourly_rows_df) | (check.pm25_valid != check.pm25_valid_df)).sum())
print(f"RDD rows: {rdd_df.hourly_rows.sum():,} | DataFrame rows: {df_counts.hourly_rows_df.sum():,} | mismatched groups: {mismatches}")
rdd_year_totals = rdd_df.groupby("year")[["hourly_rows", "pm25_valid"]].sum().reset_index()

## 9. MapReduce (Hadoop Streaming)

On our cluster we ran a real Hadoop MapReduce job on YARN. **Hadoop Streaming** lets the mapper and reducer be any program that reads lines from standard input and prints lines to standard output, so we wrote them in Python.

- **Mapper**: reads raw CSV lines, and for every reading prints `year<TAB>1,valid,pm25`. Because each station file has its own header, it re-finds the PM2.5 column whenever it meets a header line.
- **Shuffle and sort** (done by Hadoop): all lines with the same year are brought together and sorted by key.
- **Reducer**: since its input is sorted, all lines for one year arrive one after another. It keeps running totals and prints one result line each time the key changes.

`%%writefile` saves the cell's contents to a file instead of running it.

In [ ]:
%%writefile mapper.py
#!/usr/bin/env python3
import sys

pm_idx = None
for line in sys.stdin:
    line = line.rstrip("\n")
    if line.startswith("From Date"):                       # a new file's header: find PM2.5 again
        header = line.split(",")
        pm_idx = header.index("PM2.5 (ug/m3)") if "PM2.5 (ug/m3)" in header else None
        continue
    fields = line.split(",")
    year = fields[0][:4]
    if not year.isdigit():
        continue
    pm = 0.0
    valid = 0
    if pm_idx is not None and pm_idx < len(fields) and fields[pm_idx]:
        try:
            value = float(fields[pm_idx])
            if value > 0:
                pm, valid = value, 1
        except ValueError:
            pass
    print(f"{year}\t1,{valid},{pm}")                     # key <TAB> value

In [ ]:
%%writefile reducer.py
#!/usr/bin/env python3
import sys

def emit(key, rows, valid, total):
    mean = round(total / valid, 2) if valid else ""
    print(f"{key}\t{rows}\t{valid}\t{mean}")

current, rows, valid, total = None, 0, 0, 0.0
for line in sys.stdin:
    key, value = line.rstrip("\n").split("\t")
    r, v, s = value.split(",")
    if key != current:                                     # a new year starts: print the last one
        if current is not None:
            emit(current, rows, valid, total)
        current, rows, valid, total = key, 0, 0, 0.0
    rows += int(r)
    valid += int(v)
    total += float(s)
if current is not None:
    emit(current, rows, valid, total)

Colab has no Hadoop, so we run the same two scripts with the classic Unix pipe: `cat | mapper | sort | reducer`. This is exactly the local test we ran before submitting the job to the cluster, and it is what Hadoop does, just on one machine: `sort` plays the part of the shuffle. First on one file:

In [ ]:
!head -4 {DATA_DIR}/DL001.csv | python3 mapper.py      # header + 3 readings in, 3 key-value lines out
print("---")
!cat {DATA_DIR}/DL001.csv | python3 mapper.py | sort | python3 reducer.py

And on every station file. `LC_ALL=C` makes `sort` compare plain bytes, which is much faster. We write the file list to a text file and use `xargs cat` so the shell does not choke on hundreds of file names.

In [ ]:
with open("files.txt", "w") as fh:
    fh.write("\n".join(station_files))

with timed("map | sort | reduce over all files"):
    !xargs cat < files.txt | python3 mapper.py | LC_ALL=C sort -S 1G | python3 reducer.py > mr_output.tsv

mr = pd.read_csv("mr_output.tsv", sep="\t", names=["year", "hourly_rows", "pm25_valid", "mean_pm25_raw"], dtype={"year": str})
print(f"MapReduce rows: {mr.hourly_rows.sum():,}")
compare = mr.merge(rdd_year_totals, on="year", suffixes=("_mapreduce", "_rdd"))
compare["match"] = compare.hourly_rows_mapreduce == compare.hourly_rows_rdd
display(compare)

On the cluster, the same scripts were submitted to YARN with this command (from `04_Code/hadoop/run_streaming_job.sh`):

```bash
hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-*.jar \
  -D mapreduce.job.name="cpcb-pm25-by-year" \
  -D mapreduce.input.fileinputformat.split.maxsize=268435456 \
  -D mapreduce.job.reduces=2 \
  -D stream.map.input.ignoreKey=true \
  -files mapper.py,reducer.py \
  -inputformat org.apache.hadoop.mapred.lib.CombineTextInputFormat \
  -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input "/cpcb/raw/[A-Z]*.csv" -output /cpcb/mapreduce/pm25_by_year
```

- `-files` ships the two scripts to every node.
- `CombineTextInputFormat` with a 256 MB split size packs the 453 small files into **7 map tasks**. Without it Hadoop would start one task per tiny file and spend most of its time starting tasks.
- `-D mapreduce.job.reduces=2` runs two reducers in parallel.
- `ignoreKey=true` stops Hadoop from adding the byte offset to the front of each line.

Counters from our cluster run: 7 map tasks, 2 reduce tasks, 14,296,116 map input records (453 of those are header lines), **14,295,663 map output records**, about 32 seconds.

And the HDFS side, the commands we used to load and inspect the data:

```bash
hdfs dfs -mkdir -p /cpcb/raw /cpcb/reference/stations_info /cpcb/lake /cpcb/mapreduce
hdfs dfs -put -f 02_Raw_Dataset/csv/*.csv /cpcb/raw/      # copy the raw CSVs into HDFS
hdfs dfs -count -h /cpcb/raw                             # 454 files, 1.6 G
hdfs dfs -stat 'replication=%r blocksize=%o' /cpcb/raw/DL001.csv   # replication=1 blocksize=134217728
hdfs fsck /cpcb/raw                                      # Status: HEALTHY, 454 blocks
jps                                                      # NameNode, DataNode, SecondaryNameNode, ResourceManager, NodeManager
```

Replication is 1 because there is only one DataNode. Every file is smaller than the 128 MB block size, so each file is exactly one block.

## 10. Hive tables

Hive puts a SQL table layer (a *metastore*) on top of files in HDFS. Spark can act as the Hive engine, which is what `enableHiveSupport()` turned on. Here the metastore is a small Derby database on disk; on the laptop it lived next to the HDFS cluster.

Two kinds of tables:
- **EXTERNAL**: Hive only stores a pointer to files that already exist. Dropping the table keeps the files. We use these for the data our pipeline produced.
- **MANAGED**: Hive owns the files inside its warehouse folder. Dropping the table deletes the data. We use these for summary tables.

`run_sql` prints each statement with its result, so this part reads like a Hive session.

In [ ]:
WAREHOUSE = f"{LAKE_DIR}/hive_warehouse"
REF_DIR = f"{LAKE_DIR}/reference/stations_info"     # a folder with only stations_info.csv in it
os.makedirs(REF_DIR, exist_ok=True)
!cp {STATIONS_CSV} {REF_DIR}/

# Section 7 made a temporary view called "hourly". Temporary views win over real tables with
# the same name, so we drop it here, otherwise the Hive table "hourly" below would be hidden.
spark.catalog.dropTempView("hourly")

def run_sql(statement, n=20):
    print(statement.strip()); print()
    out = spark.sql(statement)
    return out.limit(n).toPandas() if out.columns else None

run_sql("DROP DATABASE IF EXISTS cpcb CASCADE")
run_sql(f"CREATE DATABASE cpcb LOCATION '{WAREHOUSE}/cpcb.db'")
spark.sql("USE cpcb")

**External table on the raw CSV.** `ROW FORMAT SERDE 'OpenCSVSerde'` tells Hive how to split each line (a SerDe is a Serializer/Deserializer). `skip.header.line.count = 1` skips the header row. Note that `LOCATION` is a *folder*, not a file: Hive reads every file in it.

In [ ]:
run_sql(f'''
CREATE EXTERNAL TABLE stations_raw (
    file_name STRING, state STRING, city STRING, agency STRING,
    station_location STRING, start_month STRING, start_month_num STRING, start_year STRING)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
STORED AS TEXTFILE
LOCATION '{REF_DIR}'
TBLPROPERTIES ('skip.header.line.count' = '1')''')
spark.sql("SELECT * FROM stations_raw LIMIT 3").show(truncate=False)

**External partitioned Parquet table on the Gold layer.** We read the column list from the Parquet files themselves and turn it into Hive types, so we do not have to type out 40 columns. `year` is left out of the column list because it is the partition column.

`MSCK REPAIR TABLE` makes Hive scan the folder and register every `year=...` sub-folder as a partition.

In [ ]:
def hive_type(t):
    return {"string": "STRING", "double": "DOUBLE", "date": "DATE", "int": "INT", "bigint": "BIGINT"}[t]

gold_schema = spark.read.parquet(GOLD_DAILY_DIR).schema
cols = ",\n    ".join(f"{f.name} {hive_type(f.dataType.simpleString())}" for f in gold_schema.fields if f.name != "year")

run_sql(f'''
CREATE EXTERNAL TABLE daily_aqi (
    {cols})
PARTITIONED BY (year INT)
STORED AS PARQUET
LOCATION '{GOLD_DAILY_DIR}' ''')
run_sql("MSCK REPAIR TABLE daily_aqi")
display(run_sql("SHOW PARTITIONS daily_aqi"))

# the hourly Silver layer as a Spark "datasource" table, also pointing at existing files
run_sql(f"CREATE TABLE hourly USING PARQUET LOCATION '{SILVER_DIR}'")
run_sql("ALTER TABLE hourly RECOVER PARTITIONS")

**Managed tables with CTAS** (`CREATE TABLE ... AS SELECT`). We store the same city-year summary twice, once as Parquet and once as ORC, so we can compare the two columnar formats. On our data, ORC came out smaller.

In [ ]:
summary_sql = '''
    SELECT d.state, d.city, d.year, COUNT(*) AS days, ROUND(AVG(d.pm25), 1) AS mean_pm25,
           ROUND(AVG(d.aqi), 0) AS mean_aqi, SUM(CASE WHEN d.aqi > 200 THEN 1 ELSE 0 END) AS poor_days
    FROM daily_aqi d WHERE d.aqi IS NOT NULL
    GROUP BY d.state, d.city, d.year'''
run_sql(f"CREATE TABLE city_year_parquet STORED AS PARQUET AS {summary_sql}")
run_sql(f"CREATE TABLE city_year_orc STORED AS ORC AS {summary_sql}")
display(run_sql("SHOW TABLES"))

# what Hive knows about each table: managed or external, file format, where the files are, size
catalog = []
for t in ["stations_raw", "daily_aqi", "hourly", "city_year_parquet", "city_year_orc"]:
    info = spark.sql(f"DESCRIBE FORMATTED {t}").toPandas()
    get = lambda key: next((v.strip() for k, v in zip(info.col_name, info.data_type) if k.strip() == key), "")
    loc = get("Location").replace("file:", "")
    catalog.append({"table": t, "type": get("Type"), "serde": get("Serde Library").rsplit(".", 1)[-1],
                    "input_format": get("InputFormat").rsplit(".", 1)[-1], "bytes": folder_bytes(loc)})
display(pd.DataFrame(catalog))

**HiveQL queries.**

- Q1: a join plus a subquery inside `WHERE`: cities whose 2022 PM2.5 is above the national 2022 mean.
- Q2: a window function, `RANK() OVER (PARTITION BY state ...)`, ranks cities inside each state, and we keep rank 1: the most polluted city in every state.
- Q3: a **Python UDF** (user-defined function). We register a normal Python function with `spark.udf.register`, and then SQL can call it like a built-in. Ours turns an AQI category into CPCB's health advice.
- Q4: `DISTRIBUTE BY` / `SORT BY`, the Hive way of saying "send rows with the same state to the same reducer, and sort inside each reducer" (unlike `ORDER BY`, which is one global sort).
- Q5: the hourly table: Delhi's monthly NO2 in 2020, where you can see the April-May lockdown dip.

In [ ]:
display(run_sql('''
SELECT c.city, c.state, c.mean_pm25, s.n_stations
FROM city_year_parquet c
JOIN (SELECT TRIM(city) AS city, COUNT(*) AS n_stations FROM stations_raw GROUP BY TRIM(city)) s
  ON c.city = s.city
WHERE c.year = 2022 AND c.days >= 200
  AND c.mean_pm25 > (SELECT AVG(pm25) FROM daily_aqi WHERE year = 2022)
ORDER BY c.mean_pm25 DESC''', n=10))

In [ ]:
display(run_sql('''
SELECT state, city, mean_pm25 FROM (
    SELECT state, city, mean_pm25,
           RANK() OVER (PARTITION BY state ORDER BY mean_pm25 DESC) AS rnk
    FROM city_year_orc WHERE year = 2022 AND days >= 200) t
WHERE rnk = 1 ORDER BY mean_pm25 DESC''', n=40))

In [ ]:
HEALTH_ADVICE = {   # CPCB AQI health-impact statements (shortened)
    "Good": "Minimal impact",
    "Satisfactory": "Minor breathing discomfort to sensitive people",
    "Moderate": "Breathing discomfort to people with lung or heart disease, children and older adults",
    "Poor": "Breathing discomfort to most people on prolonged exposure",
    "Very Poor": "Respiratory illness on prolonged exposure",
    "Severe": "Affects healthy people and seriously impacts those with existing diseases",
}

def health_advice(category):
    return HEALTH_ADVICE.get(category)

spark.udf.register("health_advice", health_advice)     # now SQL can call health_advice(...)

pd.set_option("display.max_colwidth", 90)
display(run_sql('''
SELECT aqi_category, COUNT(*) AS station_days, health_advice(aqi_category) AS advice
FROM daily_aqi WHERE year = 2022 AND city = 'Delhi' AND aqi_category IS NOT NULL
GROUP BY aqi_category ORDER BY MIN(aqi)'''))

In [ ]:
display(run_sql('''
SELECT state, city, year, mean_pm25 FROM city_year_parquet
WHERE year = 2022 DISTRIBUTE BY state SORT BY state, mean_pm25 DESC''', n=12))

display(run_sql('''
SELECT month, ROUND(AVG(no2), 1) AS mean_no2 FROM hourly
WHERE year = 2020 AND city = 'Delhi' GROUP BY month ORDER BY month'''))

## 11. Why Parquet? Speed and the query plan

### 11.1 CSV vs Parquet

The same aggregation (mean PM2.5 per station per year) on the raw CSVs and on Bronze Parquet. The CSV route has to read every byte of text, parse it and map the 198 layouts again. Parquet reads only the two or three columns the query needs, already as numbers. On our laptop Parquet was about 16 times faster (17 s vs 1 s).

In [ ]:
def bench_query(df):
    return (df.filter(F.col("pm25").isNotNull())
              .groupBy("station_id", F.year("ts").alias("y")).agg(F.avg("pm25")).count())

raw_again = reduce(lambda a, b: a.unionByName(b), [build_group_df(h, f) for h, f in groups.items()])
bench = []
for label, frame in [("Raw CSV", raw_again), ("Bronze Parquet", spark.read.parquet(BRONZE_DIR))]:
    t0 = time.perf_counter()
    bench_query(frame)
    bench.append({"source": label, "seconds": round(time.perf_counter() - t0, 2)})
bench = pd.DataFrame(bench)
display(bench)
print(f"Parquet is {bench.seconds[0] / bench.seconds[1]:.1f}x faster here")

### 11.2 Storage at each layer

In [ ]:
sizes = pd.DataFrame([{"layer": "raw CSV", "MB": round(raw_bytes / 1e6, 1)}] + [{"layer": name, "MB": round(folder_bytes(p) / 1e6, 1)} for name, p in
                      [("Bronze (hourly, all rows)", BRONZE_DIR),
                       ("Silver (hourly, cleaned)", SILVER_DIR), ("Gold (daily AQI)", GOLD_DAILY_DIR)]])
display(sizes)

### 11.3 Reading the physical plan

`explain("formatted")` shows what Spark will actually do. Two things to point out:

- **PartitionFilters: year = 2022**. Spark opens only the `year=2022` folder and skips the other 13 years without reading them.
- **BroadcastHashJoin**. The small stations table is broadcast to every worker, so the large table never has to be shuffled for the join.

In [ ]:
# "daily" is still cached in memory from Section 7, and a cached table hides the file scan.
# The analysis is done, so we clear the cache and Spark goes back to reading the Parquet files.
spark.catalog.clearCache()
spark.sql('''SELECT d.state, AVG(d.pm25) AS mean_pm25
             FROM daily d JOIN stations s USING (station_id)
             WHERE d.year = 2022 GROUP BY d.state''').explain("formatted")

## Wrap-up

What this notebook did, end to end:

- **Ingest**: 453 station files with 198 header layouts and mixed units turned into one consistent table of 14.3 million hourly rows (Bronze Parquet, about 0.5 GB instead of 1.7 GB).
- **Clean**: seven logged rules. About 2.2 million empty rows dropped and about 2.8 million bad values nulled, leaving 11.96 million clean rows (Silver).
- **AQI**: CPCB's official daily AQI per station with window functions (Gold, about 522 thousand station-days).
- **Answers** (full data):
  - 91 of 133 well-monitored cities are above India's annual PM2.5 limit of 40; the worst 15 are all in north India, led by Ghaziabad (108) and Delhi (103).
  - In November the Indo-Gangetic Plain averages about 2.7 times the PM2.5 of the rest of India.
  - PM2.5 is worst on winter nights. In Delhi it jumps from about 180 to about 296 ug/m3 on the day after Diwali, with an hourly peak near 600 around midnight.
  - During the 2020 lockdown, NO2 fell about 45% and PM2.5 about 41% compared with 2019 at the same stations.
  - At the same 97 stations, average PM2.5 fell about 16% from 2018 to 2022.
- **Cross-checks**: MapReduce, RDDs and DataFrames all count exactly the same 14,295,663 rows.

In [ ]:
spark.stop()